# Clean and Merge Coursera Course Datasets

This notebook creates **one final weighted CSV**: `merged_core_it_courses_fixed.csv`. It keeps core IT courses, keeps semi-IT courses with lower weight to improve recall, removes clear non-IT noise, fixes missing values, and removes duplicates.

In [ ]:
import pandas as pd
import re

df1 = pd.read_csv("Coursera_IT_cleaned.csv")
df2 = pd.read_csv("coursera_course_dataset_v2_no_null.csv")

print("Dataset 1:", df1.shape)
print("Dataset 2:", df2.shape)



df1_clean = pd.DataFrame({
    "course_title": df1.get("Course Name", ""),
    "provider": df1.get("University", ""),
    "level": df1.get("Difficulty Level", ""),
    "rating": df1.get("Course Rating", ""),
    "reviews": "",
    "url": df1.get("Course URL", ""),
    "description": df1.get("Course Description", ""),
    "skills": df1.get("Skills", ""),
    "source_dataset": "Coursera_IT_cleaned"
})

df2_clean = pd.DataFrame({
    "course_title": df2.get("Title", ""),
    "provider": df2.get("Organization", ""),
    "level": df2.get("Metadata", ""),
    "rating": df2.get("Ratings", ""),
    "reviews": df2.get("Review counts", ""),
    "url": "",
    "description": "",
    "skills": df2.get("Skills", ""),
    "source_dataset": "coursera_course_dataset_v2_no_null"
})

merged = pd.concat([df1_clean, df2_clean], ignore_index=True)


def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).replace("�", " ")
    x = re.sub(r"\s+", " ", x)
    return x.strip()

for col in merged.columns:
    merged[col] = merged[col].apply(clean_text)


def clean_level(x):
    x = str(x).lower()
    if "beginner" in x:
        return "Beginner"
    if "intermediate" in x:
        return "Intermediate"
    if "advanced" in x:
        return "Advanced"
    if "mixed" in x or "all levels" in x:
        return "All Levels"
    return "Unknown"

merged["level"] = merged["level"].apply(clean_level)


def clean_skills(x):
    x = str(x).lower()
    x = x.replace("[", "").replace("]", "")
    x = x.replace("'", "").replace('"', "")
    x = x.replace(";", ",")
    x = re.sub(r"\s+", " ", x)

    parts = re.split(r",|\|", x)
    parts = [p.strip() for p in parts if p.strip()]

    seen = set()
    clean_parts = []
    for p in parts:
        if p not in seen:
            seen.add(p)
            clean_parts.append(p)

    return ", ".join(clean_parts)

merged["skills"] = merged["skills"].apply(clean_skills)


core_it_keywords = [
    "python", "java", "javascript", "c++", "c#", "sql", "database",
    "programming", "coding", "software", "web", "backend", "frontend",
    "full stack", "cloud", "aws", "azure", "google cloud", "devops",
    "cybersecurity", "security", "network", "linux", "operating system",
    "data science", "machine learning", "deep learning", "artificial intelligence",
    "computer vision", "nlp", "data analysis", "data analytics", "data visualization",
    "big data", "computer science", "information technology", "algorithm",
    "data structure", "html", "css", "react", "node", "django", "flask",
    "tensorflow", "pytorch", "api", "git", "github", "selenium", "testing", "automation",
    "business intelligence", "power bi", "tableau", "excel analytics", "agile", "scrum",
    "project management", "mobile development", "flutter", "android", "ios"
]

semi_it_keywords = [
    "business analytics", "business intelligence", "digital transformation",
    "fintech", "health informatics", "bioinformatics", "ai in healthcare",
    "marketing analytics", "financial analytics", "data for business",
    "google sheets", "google slides", "spreadsheet", "excel",
    "project management", "agile", "scrum", "product management",
    "data storytelling", "dashboard", "visualization"
]

strong_non_it_keywords = [
    "music", "history", "philosophy", "religion", "screenplay", "film",
    "language learning", "grammar", "writing", "nutrition", "law",
    "public health", "medicine", "biology", "psychology", "art history",
    "social sciences"
]

soft_noise_keywords = [
    "marketing", "finance", "accounting", "management", "leadership", "business"
]

def classify_course(row):
    text = (
        row["course_title"] + " " +
        row["skills"] + " " +
        row["description"]
    ).lower()

    has_core = any(k in text for k in core_it_keywords)
    has_semi = any(k in text for k in semi_it_keywords)
    has_strong_non_it = any(k in text for k in strong_non_it_keywords)
    has_soft_noise = any(k in text for k in soft_noise_keywords)

    if has_core and not has_strong_non_it and not has_soft_noise:
        return "core_it"
    if has_core and (has_soft_noise or has_semi) and not has_strong_non_it:
        return "semi_it"
    if has_semi and not has_strong_non_it:
        return "semi_it"
    return "non_it"

merged["it_label"] = merged.apply(classify_course, axis=1)


merged = merged[merged["it_label"].isin(["core_it", "semi_it"])].copy()


def assign_quality_weight(row):
    if row["it_label"] == "semi_it":
        return 0.70
    if row["source_dataset"] == "Coursera_IT_cleaned":
        return 1.00
    return 0.90

merged["quality_weight"] = merged.apply(assign_quality_weight, axis=1)

for col in ["course_title", "provider", "level", "url", "description", "skills", "source_dataset", "it_label"]:
    merged[col] = merged[col].fillna("").astype(str).str.strip()

merged["description"] = merged.apply(
    lambda row: row["description"] if row["description"].strip()
    else f"{row['course_title']} covers {row['skills']}",
    axis=1
)

merged["rating"] = pd.to_numeric(merged["rating"], errors="coerce").fillna(0)
merged["reviews"] = pd.to_numeric(merged["reviews"], errors="coerce").fillna(0)
merged["level"] = merged["level"].replace("", "Unknown").fillna("Unknown")


def normalize_title(title):
    title = str(title).lower()
    title = re.sub(r"[^a-z0-9 ]", " ", title)
    title = re.sub(r"\s+", " ", title).strip()

    remove_words = [
        "course", "specialization", "professional certificate",
        "project", "guided project", "certificate"
    ]
    for word in remove_words:
        title = title.replace(word, "")

    title = re.sub(r"\s+", " ", title).strip()
    return title

merged["normalized_title"] = merged["course_title"].apply(normalize_title)

merged = merged.sort_values(
    by=["quality_weight", "rating", "reviews"],
    ascending=[False, False, False]
)

merged = merged.drop_duplicates(subset=["normalized_title"], keep="first")
merged = merged.drop(columns=["normalized_title"])


final_columns = [
    "course_title", "provider", "level", "rating", "reviews", "url",
    "description", "skills", "source_dataset", "it_label", "quality_weight"
]

merged = merged[final_columns]

output_path = "merged_core_it_courses_fixed.csv"
merged.to_csv(output_path, index=False)

print("DONE")
print("Final weighted IT dataset:", merged.shape)
print("Core IT rows:", (merged["it_label"] == "core_it").sum())
print("Semi-IT rows kept with lower weight:", (merged["it_label"] == "semi_it").sum())
print("Saved file:", output_path)

print("\nMissing values after final cleanup:")
print(merged.isna().sum())

display(merged.head())

Dataset 1: (1256, 7)
Dataset 2: (623, 7)
DONE
Final weighted IT dataset: (1474, 11)
Core IT rows: 924
Semi-IT rows kept with lower weight: 550
Saved file: merged_core_it_courses_fixed.csv

Missing values after final cleanup:
course_title      0
provider          0
level             0
rating            0
reviews           0
url               0
description       0
skills            0
source_dataset    0
it_label          0
quality_weight    0
dtype: int64


,course_title,provider,level,rating,reviews,url,description,skills,source_dataset,it_label,quality_weight
26,Create an FPS Weapon in Unity (Part 1 - Revolver),Coursera Project Network,Advanced,5.0,0.0,https://www.coursera.org/learn/create-fps-weap...,"In this one-hour, project-based course, you'll...",gamification of learning project process .prop...,Coursera_IT_cleaned,core_it,1.0
104,Encryption And Decryption Using C++,Coursera Project Network,Beginner,5.0,0.0,https://www.coursera.org/learn/encryption-and-...,"In this 2-hour long project-based course, you ...",project cryptography project mine algorithms l...,Coursera_IT_cleaned,core_it,1.0
209,Compare time series predictions of COVID-19 de...,Coursera Project Network,Beginner,5.0,0.0,https://www.coursera.org/learn/compare-time-se...,"In this 2-hour long project-based course, you ...",python programming modeling analysis project t...,Coursera_IT_cleaned,core_it,1.0
220,JavaScript Strings: Properties and Methods,Coursera Project Network,Beginner,5.0,0.0,https://www.coursera.org/learn/javascript-stri...,In this beginning-level course you will learn ...,.properties hold project concatenation ternary...,Coursera_IT_cleaned,core_it,1.0
221,"Java Inheritance, Composition and Aggregation",Coursera Project Network,Beginner,5.0,0.0,https://www.coursera.org/learn/java-inheritanc...,In this project you will create a Java applica...,child is-a computer programming parent ordered...,Coursera_IT_cleaned,core_it,1.0
